# Store Sales – Data Pre-processing

Implements the preprocessing plan from **§8 of `01_eda.ipynb`**. Reads the raw csvs from `data/`, writes model-ready feature tables to `data/processed/`.

| step | plan item (§8) |
|---|---|
| 1. Holiday table parsing | transfers, Work Days, Bridges, de-duplication, locale→store mapping, earthquake flag from "Terremoto Manabi" events |
| 2. Calendar features | day-of-week/month, payday flags, days to/since national holiday, Dec-25 proximity |
| 3. Oil | reindex to full calendar + ffill (no interpolation — it peeks at future prices) |
| 4. Panel assembly | reinsert the 4 missing Dec-25 dates with `sales = 0` **before** any lag is built |
| 5. Lag & rolling features | per-(store, family) lags 16/21/28, rolling mean/std 7/28 on the lag-16 base, dead-series flag, lagged `transactions` |
| 6. Merges | stores, oil, calendar, holidays onto the panel |
| 7. Filtering & target | drop pre-opening rows, drop reinserted Dec-25 rows again, drop rows without full lag history, `log1p` target |
| 8. Dtypes, checks & save | native categoricals, leak/NaN assertions, save |

**Design rule:** every sales/transactions-derived feature uses a shift of **≥ 16 days**, so all features are computable for the entire 16-day test horizon (2017-08-16 → 2017-08-31) without recursion.

## 0. Setup & load

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

DATA = Path('../data')
OUT = DATA / 'processed'
OUT.mkdir(exist_ok=True)

train = pd.read_csv(DATA / 'train.csv', parse_dates=['date'])
test = pd.read_csv(DATA / 'test.csv', parse_dates=['date'])
stores = pd.read_csv(DATA / 'stores.csv')
oil = pd.read_csv(DATA / 'oil.csv', parse_dates=['date'])
holidays = pd.read_csv(DATA / 'holidays_events.csv', parse_dates=['date'])
transactions = pd.read_csv(DATA / 'transactions.csv', parse_dates=['date'])

for df in (train, test, oil, holidays, transactions):
    df['date'] = df['date'].astype('datetime64[ns]')

CAL = pd.date_range('2013-01-01', '2017-08-31', freq='D')  # train start -> test end
print(f'train {train.shape}, test {test.shape}, calendar {len(CAL)} days')

train (3000888, 6), test (28512, 5), calendar 1704 days


## 1. Holiday table parsing

Rules from the EDA:
- rows with `transferred = True` are days where the named holiday was **not** celebrated → drop;
- `type = Transfer` rows hold the date it **was** celebrated → treat as Holiday;
- `Additional` and `Bridge` are extra days off → holiday; `Work Day` compensates a Bridge → explicitly *not* a day off;
- `Event` rows are not days off → separate flag; the "Terremoto Manabi" events (2016-04-16 + ~1 month) provide the earthquake flag for free;
- `locale` maps National → all stores, Regional → `stores.state`, Local → `stores.city`; 38 dates carry >1 record → de-duplicate per (date, store).

In [2]:
hol = holidays[~holidays['transferred']].copy()      # transferred-out: not celebrated that day
hol['type'] = hol['type'].replace({'Transfer': 'Holiday'})  # actual celebration dates

celebrated = hol[hol['type'].isin(['Holiday', 'Additional', 'Bridge'])]
workday = hol[hol['type'] == 'Work Day']
events = hol[hol['type'] == 'Event']
earthquake = events[events['description'].str.startswith('Terremoto')]

print(f'celebrated: {len(celebrated)}, workdays: {len(workday)}, '
      f'events: {len(events)} (earthquake: {len(earthquake)})')

celebrated: 277, workdays: 5, events: 56 (earthquake: 31)


In [3]:
def to_store_days(df):
    """Map holiday rows to the (date, store_nbr) pairs they apply to."""
    nat = df.loc[df['locale'] == 'National', ['date']].merge(stores[['store_nbr']], how='cross')
    reg = df[df['locale'] == 'Regional'].merge(
        stores[['store_nbr', 'state']], left_on='locale_name', right_on='state')
    loc = df[df['locale'] == 'Local'].merge(
        stores[['store_nbr', 'city']], left_on='locale_name', right_on='city')
    out = pd.concat([nat[['date', 'store_nbr']], reg[['date', 'store_nbr']],
                     loc[['date', 'store_nbr']]])
    return out.drop_duplicates()  # de-duplicate dates carrying >1 record

flag_sources = {'is_holiday': celebrated, 'is_workday': workday,
                'is_event': events, 'is_earthquake': earthquake}
hol_store = None
for name, df in flag_sources.items():
    sd = to_store_days(df).assign(**{name: 1})
    hol_store = sd if hol_store is None else hol_store.merge(sd, on=['date', 'store_nbr'], how='outer')
hol_store = hol_store.fillna(0).astype({c: 'int8' for c in flag_sources})
print(f'{len(hol_store):,} (date, store) holiday rows')
hol_store.sample(5, random_state=0)

9,139 (date, store) holiday rows


,date,store_nbr,is_holiday,is_workday,is_event,is_earthquake
6778,2016-05-10,11,0,0,1,1
3981,2015-01-02,11,1,0,0,0
1652,2013-12-23,34,1,0,0,0
888,2013-02-12,6,1,0,0,0
6654,2016-05-07,49,1,0,1,1


## 2. Calendar features (date level)

In [4]:
cal = pd.DataFrame({'date': CAL})

cal['dayofweek'] = cal['date'].dt.dayofweek.astype('int8')
cal['day_of_month'] = cal['date'].dt.day.astype('int8')
cal['month'] = cal['date'].dt.month.astype('int8')
cal['year'] = cal['date'].dt.year.astype('int16')

dom = cal['date'].dt.day
cal['is_payday'] = (dom.isin([15, 16, 17]) | dom.isin([1, 2])
                    | cal['date'].dt.is_month_end).astype('int8')

# distance to/from the nearest *national* celebrated holiday
nat_dates = np.sort(pd.DatetimeIndex(
    celebrated.loc[celebrated['locale'] == 'National', 'date'].unique()).values)
d = cal['date'].values
nxt = np.searchsorted(nat_dates, d, side='left')
prv = np.searchsorted(nat_dates, d, side='right') - 1
assert nxt.max() < len(nat_dates) and prv.min() >= 0  # holiday table spans 2012->2017-12
cal['days_to_nat_holiday'] = ((nat_dates[nxt] - d) / np.timedelta64(1, 'D')).astype('int16')
cal['days_since_nat_holiday'] = ((d - nat_dates[prv]) / np.timedelta64(1, 'D')).astype('int16')

# Dec-25 proximity (wraps over New Year)
dec25 = pd.to_datetime(cal['date'].dt.year.astype(str) + '-12-25')
dd = (dec25 - cal['date']).dt.days
cal['days_to_dec25'] = np.where(dd >= 0, dd, dd + 365).astype('int16')
cal.head()

,date,dayofweek,day_of_month,month,year,is_payday,days_to_nat_holiday,days_since_nat_holiday,days_to_dec25
0,2013-01-01,1,1,1,2013,1,0,0,358
1,2013-01-02,2,2,1,2013,1,40,1,357
2,2013-01-03,3,3,1,2013,0,39,2,356
3,2013-01-04,4,4,1,2013,0,38,3,355
4,2013-01-05,5,5,1,2013,0,37,4,354


## 3. Oil

Forward-fill over the full calendar — **no interpolation**, which would peek at future prices. `bfill` only fills the very first day(s).

In [5]:
oil_full = (oil.set_index('date')['dcoilwtico'].reindex(CAL)
            .ffill().bfill()
            .rename('oil_price').rename_axis('date').reset_index())
assert oil_full['oil_price'].notna().all()
oil_full.head(3)

,date,oil_price
0,2013-01-01,93.14
1,2013-01-02,93.14
2,2013-01-03,92.97


## 4. Panel assembly

Reinsert the 4 absent Dec-25 dates with `sales = 0` (stores closed — zero is the true value) **before** building lags, otherwise a row-based shift silently misaligns every lag across the Christmas boundary. The reinserted rows are tagged `id = -1` so they can be dropped again after the lag computation. Train and test are concatenated so test rows get identical features.

In [6]:
train_cal = pd.date_range(train['date'].min(), train['date'].max(), freq='D')
missing_dates = train_cal.difference(train['date'].unique())
print('reinserting:', [str(x.date()) for x in missing_dates])

pairs = train[['store_nbr', 'family']].drop_duplicates()
reinserted = (pd.DataFrame({'date': missing_dates}).merge(pairs, how='cross')
              .assign(id=-1, sales=0.0, onpromotion=0))

panel = pd.concat([train, reinserted, test.assign(sales=np.nan)], ignore_index=True)
panel = panel.sort_values(['store_nbr', 'family', 'date'], ignore_index=True)
assert len(panel) == len(CAL) * len(pairs), 'panel must be a complete daily grid'
print(f'panel: {len(panel):,} rows = {len(CAL)} days x {len(pairs)} (store, family) pairs')

reinserting: ['2013-12-25', '2014-12-25', '2015-12-25', '2016-12-25']


panel: 3,036,528 rows = 1704 days x 1782 (store, family) pairs


## 5. Lag & rolling features

Computed on a wide (date × series) matrix — with a complete grid, `shift`/`rolling` over rows are exact day-based windows and group boundaries cannot leak into each other. The lag-16 base is the most recent sales value available on **every** test day, so rolling stats on it are leak-free for the whole horizon. `is_dead_series` flags pairs whose last 90 observable days are all zero (the §8 "predict 0" rule — applied at modelling time).

In [7]:
sales_wide = panel.pivot(index='date', columns=['store_nbr', 'family'], values='sales')
base = sales_wide.shift(16)  # most recent sales available at any test date

feats = {
    'sales_lag_16': base,
    'sales_lag_21': sales_wide.shift(21),
    'sales_lag_28': sales_wide.shift(28),
    'sales_lag16_roll7_mean': base.rolling(7).mean(),
    'sales_lag16_roll28_mean': base.rolling(28).mean(),
    'sales_lag16_roll7_std': base.rolling(7).std(),
    'sales_lag16_roll28_std': base.rolling(28).std(),
    # yearly seasonality: same weekday one year back (364 = 52 weeks), plus the
    # 7-day window centred on the same calendar date last year (shift 361 -> d-367..d-361)
    'sales_lag_364': sales_wide.shift(364),
    'sales_lag364_roll7_mean': sales_wide.shift(361).rolling(7).mean(),
    # same-weekday mean over the last 4 observable weeks (21/28/35/42 are all multiples of 7)
    'sales_dow4_mean': (sales_wide.shift(21) + sales_wide.shift(28)
                        + sales_wide.shift(35) + sales_wide.shift(42)) / 4,
    # intermittency: share of zero-sales days in the last 28 observable days
    'sales_zero_rate_28': (base == 0).astype('float32').rolling(28).mean(),
    'is_dead_series': base.rolling(90, min_periods=90).sum() == 0,
}
lag_long = pd.concat(feats, axis=1).stack([1, 2]).reset_index()

float_cols = [c for c in feats if c != 'is_dead_series']
lag_long[float_cols] = lag_long[float_cols].astype('float32')
lag_long['is_dead_series'] = lag_long['is_dead_series'].astype('int8')

panel = panel.merge(lag_long, on=['date', 'store_nbr', 'family'], how='left')

# promotion intensity: onpromotion is future-known (test.csv includes it), so the
# windows may end at the current day -- no shift needed, same-day value is already a column
promo_wide = panel.pivot(index='date', columns=['store_nbr', 'family'],
                         values='onpromotion').astype('float32')
promo_long = pd.concat({
    'promo_roll7_mean': promo_wide.rolling(7).mean(),
    'promo_roll28_mean': promo_wide.rolling(28).mean(),
}, axis=1).stack([1, 2]).reset_index()
promo_cols = ['promo_roll7_mean', 'promo_roll28_mean']
promo_long[promo_cols] = promo_long[promo_cols].astype('float32')
panel = panel.merge(promo_long, on=['date', 'store_nbr', 'family'], how='left')

print(f'dead-series share: {lag_long["is_dead_series"].mean():.1%}')

/var/folders/gy/18t4md_j3q39g8dz2229hbnc0000gn/T/ipykernel_27857/3907343163.py:23: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  lag_long = pd.concat(feats, axis=1).stack([1, 2]).reset_index()


/var/folders/gy/18t4md_j3q39g8dz2229hbnc0000gn/T/ipykernel_27857/3907343163.py:38: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  }, axis=1).stack([1, 2]).reset_index()


dead-series share: 17.5%


In [8]:
# transactions: zero test coverage -> lag-16 features only (per store)
# fillna(0): closed / pre-opening days truly had 0 transactions; the NaN test region
# never enters any feature because every shift is >= the 16-day horizon.
tx_wide = (transactions.pivot(index='date', columns='store_nbr', values='transactions')
           .reindex(CAL).rename_axis('date').fillna(0))
tx_feats = {
    'tx_lag_16': tx_wide.shift(16),
    'tx_lag16_roll7_mean': tx_wide.shift(16).rolling(7).mean(),
    'tx_lag16_roll28_mean': tx_wide.shift(16).rolling(28).mean(),
}
tx_long = pd.concat(tx_feats, axis=1).stack().reset_index()
tx_long[list(tx_feats)] = tx_long[list(tx_feats)].astype('float32')

panel = panel.merge(tx_long, on=['date', 'store_nbr'], how='left')
panel[['date', 'store_nbr', 'tx_lag_16', 'tx_lag16_roll7_mean']].tail(3)

/var/folders/gy/18t4md_j3q39g8dz2229hbnc0000gn/T/ipykernel_27857/813845501.py:11: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  tx_long = pd.concat(tx_feats, axis=1).stack().reset_index()


,date,store_nbr,tx_lag_16,tx_lag16_roll7_mean
3036525,2017-08-29,54,1054.0,819.571411
3036526,2017-08-30,54,818.0,816.142883
3036527,2017-08-31,54,802.0,823.285706


## 6. Merge stores, oil, calendar, holidays onto the panel

In [9]:
panel = (panel
         .merge(stores.rename(columns={'type': 'store_type'}), on='store_nbr', how='left')
         .merge(oil_full, on='date', how='left')
         .merge(cal, on='date', how='left')
         .merge(hol_store, on=['date', 'store_nbr'], how='left'))

hol_cols = ['is_holiday', 'is_workday', 'is_event', 'is_earthquake']
panel[hol_cols] = panel[hol_cols].fillna(0).astype('int8')
assert len(panel) == len(CAL) * len(pairs), 'merges must not duplicate rows'
panel.head(3)

,id,date,store_nbr,family,sales,onpromotion,sales_lag_16,sales_lag_21,sales_lag_28,sales_lag16_roll7_mean,...,month,year,is_payday,days_to_nat_holiday,days_since_nat_holiday,days_to_dec25,is_holiday,is_workday,is_event,is_earthquake
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,NaN,NaN,NaN,NaN,...,1,2013,1,0,0,358,1,0,0,0
1,1782,2013-01-02,1,AUTOMOTIVE,2.0,0,NaN,NaN,NaN,NaN,...,1,2013,1,40,1,357,0,0,0,0
2,3564,2013-01-03,1,AUTOMOTIVE,3.0,0,NaN,NaN,NaN,NaN,...,1,2013,0,39,2,356,0,0,0,0


## 7. Filtering & target

- drop rows before each store's first sale (pre-opening zeros mean "store did not exist", not "zero demand");
- drop the reinserted Dec-25 rows again — they were only needed for lag alignment;
- drop early-2013 rows without a full lag history (first 43 days);
- target: `log1p(sales)` → RMSLE becomes plain RMSE.

The plan's "truncate training history to 2015+" item is an **experiment, not preprocessing** — full history is kept here and the cut is a one-liner at modelling time.

In [10]:
first_sale = (train.loc[train['sales'] > 0].groupby('store_nbr')['date'].min()
              .rename('first_sale_date'))
panel = panel.merge(first_sale, on='store_nbr', how='left')
pre_opening = panel['date'] < panel['first_sale_date']
print(f'pre-opening rows dropped: {pre_opening.sum():,}')
panel = panel.loc[~pre_opening].drop(columns='first_sale_date')

is_test = panel['date'] >= test['date'].min()
train_out = panel[~is_test & (panel['id'] != -1)].copy()  # id == -1: reinserted Dec-25 rows
test_out = panel[is_test].drop(columns='sales')

# yearly (364) lags are NaN for the whole first year and for young stores --
# keep those rows, LightGBM handles NaN natively; dropna only on the short lags
lag_cols = [c for c in panel.columns
            if c.startswith(('sales_lag', 'tx_lag')) and '364' not in c]
n0 = len(train_out)
train_out = train_out.dropna(subset=lag_cols)
print(f'rows without full short-lag history dropped: {n0 - len(train_out):,}')
print(f'NaN share in sales_lag_364 (train): {train_out["sales_lag_364"].isna().mean():.1%}')

train_out['sales_log1p'] = np.log1p(train_out['sales']).astype('float32')

pre-opening rows dropped: 222,552


rows without full short-lag history dropped: 63,789
NaN share in sales_lag_364 (train): 18.2%


## 8. Dtypes, checks & save

Native `category` dtype for the categoricals (LightGBM consumes it directly; no scaling needed for trees). Saved as **pickle** — preserves dtypes incl. categories; this env has no `pyarrow`, switch to parquet if it gets installed.

In [11]:
cat_cols = ['store_nbr', 'family', 'city', 'state', 'store_type', 'cluster']
for df in (train_out, test_out):
    df[cat_cols] = df[cat_cols].astype('category')
    df['onpromotion'] = df['onpromotion'].astype('int16')

# --- leak / completeness checks ---
yearly_cols = ['sales_lag_364', 'sales_lag364_roll7_mean']  # NaN allowed (first year / young stores)
assert len(test_out) == len(test) == 28512
assert test_out.drop(columns='id').notna().all().all(), 'test features must be complete'
assert train_out.drop(columns=yearly_cols).notna().all().all(), 'train must contain no NaN'
assert train_out['date'].max() == pd.Timestamp('2017-08-15')
assert test_out['date'].min() == pd.Timestamp('2017-08-16')
assert (train_out.loc[train_out['is_dead_series'] == 1, 'sales_lag16_roll28_mean'] == 0).all()

train_out.to_pickle(OUT / 'train_features.pkl')
test_out.to_pickle(OUT / 'test_features.pkl')
print(f'train_features: {train_out.shape}, test_features: {test_out.shape}')
print(f'written to {OUT}/')

train_features: (2715042, 41), test_features: (28512, 39)
written to data/processed/


In [12]:
summary = pd.DataFrame({'dtype': train_out.dtypes.astype(str),
                        'n_unique': train_out.nunique()})
summary

,dtype,n_unique
id,int64,2715042
date,datetime64[ns],1641
store_nbr,category,54
family,category,33
sales,float64,374539
onpromotion,int16,362
sales_lag_16,float32,372192
sales_lag_21,float32,371779
sales_lag_28,float32,371150
sales_lag16_roll7_mean,float32,522002


## Output summary

`data/processed/train_features.pkl` and `test_features.pkl`, one row per (date, store, family):

- **keys / target:** `id`, `date`, `store_nbr`, `family`, `sales` + `sales_log1p` (train only)
- **known covariates:** `onpromotion`, `oil_price`, `promo_roll7/28_mean` (promotion intensity — future-known, windows end at the current day)
- **store metadata:** `city`, `state`, `store_type`, `cluster`
- **calendar:** `dayofweek`, `day_of_month`, `month`, `year`, `is_payday`, `days_to_nat_holiday`, `days_since_nat_holiday`, `days_to_dec25`
- **holidays:** `is_holiday`, `is_workday`, `is_event`, `is_earthquake`
- **history (all shifts ≥ 16 d, leak-free over the full horizon):** `sales_lag_16/21/28`, `sales_lag16_roll7/28_mean`, `sales_lag16_roll7/28_std`, `tx_lag_16`, `tx_lag16_roll7/28_mean`, `is_dead_series`
- **weekday & intermittency aggregates:** `sales_dow4_mean` (same-weekday mean of the last 4 observable weeks), `sales_zero_rate_28` (share of zero days in the last 28 observable days)
- **yearly seasonality:** `sales_lag_364` (same weekday last year), `sales_lag364_roll7_mean` (±3-day window around the same date last year) — NaN for the first year of history and for young stores; LightGBM handles NaN natively

Next (modelling notebook): time-based validation (last 16 days + rolling folds), naive lag-7 baselines, the `is_dead_series` → predict-0 rule, the 2015+ truncation experiment, then a global LightGBM (direct strategy).